In [1]:
!pip install --upgrade protobuf==3.20.*
!pip install --upgrade transformers accelerate tokenizers
!pip install --upgrade sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tens

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModel,
    pipeline, get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgb
from tqdm.auto import tqdm

2025-11-20 13:43:07.752482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763646187.941387      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763646187.993847      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Backbones & models (as you requested)
BACKBONE = "vinai/phobert-base"
NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
NER_MODEL_NAME = "undertheseanlp/vietnamese-ner-v1.4.0a2"

# Dataset paths (based on your uploaded Kaggle datasets)
TRAIN_CSV = "/kaggle/input/vihallu-dataset/vihallu-train.csv"
WARMUP_CSV = "/kaggle/input/vihallu-dataset/vihallu-warmup.csv"  # if exists / optional
PRIVATE_TEST_CSV = "/kaggle/input/vihallu-private-dataset/vihallu-private-test.csv"
VISRAL_TRAIN_PROBS_CSV = "/kaggle/input/vihallu-vistral-dataset/vistral_train_predictions_with_probs.csv"
VISRAL_TEST_PROBS_CSV = "/kaggle/input/vihallu-vistral-dataset/vistral_testtt_predictions_with_probs.csv"


In [4]:
# Training hyperparams (kept as in your original file)
BATCH_SIZE = 16
MAX_LENGTH = 256
LEARNING_RATE = 1.0111624625603285e-05
EPOCHS = 7
WEIGHT_DECAY = 0.010571279628255452
LGBM_MODEL_PATH_TEMPLATE = "/kaggle/working/lgbm_final_fold_{}.txt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
df = pd.read_csv(TRAIN_CSV)
# combine text parts exactly as your original script
df['text'] = df['context'].fillna("") + " " + df['prompt'].fillna("") + " " + df['response'].fillna("")
labels, label_names = pd.factorize(df['label'])
df['label_id'] = labels
num_labels = len(label_names)

In [6]:
class HallucinationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        encoding = self.tokenizer.encode_plus(
            str(self.texts[item]),
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

In [7]:
# ---------------- STEP 1: FINE-TUNE PHOBERT ----------------
# split
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

# model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=num_labels)
model.to(device)

train_dataset = HallucinationDataset(df_train.text.to_numpy(), df_train.label_id.to_numpy(), tokenizer, MAX_LENGTH)
val_dataset = HallucinationDataset(df_val.text.to_numpy(), df_val.label_id.to_numpy(), tokenizer, MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * EPOCHS)

best_f1 = 0.0
for epoch in range(EPOCHS):
    print(f'======== Epoch {epoch + 1}/{EPOCHS} ========')
    model.train()
    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            )
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    print(classification_report(all_labels, all_preds, target_names=label_names))
    print(f"Validation Macro-F1: {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        print("New best model found! Saving to disk.")
        model.save_pretrained("/kaggle/working/phobert_finetuned_model")
        tokenizer.save_pretrained("/kaggle/working/phobert_finetuned_model")

print("Finished fine-tuning. Best model saved to /kaggle/working/phobert_finetuned_model")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

...

Evaluating:   0%|          | 0/88 [00:00<?, ?it/s]

              precision    recall  f1-score   support

   extrinsic       0.56      0.58      0.57       461
          no       0.53      0.63      0.58       449
   intrinsic       0.54      0.43      0.48       490

    accuracy                           0.55      1400
   macro avg       0.55      0.55      0.54      1400
weighted avg       0.55      0.55      0.54      1400

Validation Macro-F1: 0.5431
New best model found! Saving to disk.
======== Epoch 7/7 ========


Training:   0%|          | 0/350 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/88 [00:00<?, ?it/s]

              precision    recall  f1-score   support

   extrinsic       0.55      0.56      0.56       461
          no       0.52      0.63      0.57       449
   intrinsic       0.53      0.42      0.47       490

    accuracy                           0.54      1400
   macro avg       0.54      0.54      0.53      1400
weighted avg       0.54      0.54      0.53      1400

Validation Macro-F1: 0.5331
Finished fine-tuning. Best model saved to /kaggle/working/phobert_finetuned_model


In [8]:
# ---------------- STEP 2: FEATURE ENGINEERING + FINAL TRAINING ----------------
FINETUNED_BACKBONE_PATH = "/kaggle/working/phobert_finetuned_model"

BATCH_SIZE_FE = 16  # keep same batch size for feature extraction

# helper funcs
def extract_embeddings(texts, model_emb, tokenizer_emb, desc, batch_size=BATCH_SIZE_FE):
    X_parts = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i + batch_size]
            enc = tokenizer_emb(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
            outputs = model_emb(**{k: v.to(device) for k, v in enc.items()})
            # mean pooling on last_hidden_state
            X_parts.append(outputs.last_hidden_state.mean(dim=1).cpu().numpy())
    return np.vstack(X_parts) if len(X_parts) else np.zeros((0, model_emb.config.hidden_size))

def feat_engineer(df_local):
    f = pd.DataFrame()
    for col in ["context", "prompt", "response"]:
        s = df_local[col].fillna("").astype(str)
        f[f"{col}_len"], f[f"{col}_words"] = s.str.len(), s.str.split().apply(len)
    return f.values

def get_entity_set(text, pipe):
    if not isinstance(text, str) or not text.strip():
        return set()
    try:
        entities = pipe(text, grouped_entities=True)
        return {entity['word'].replace(" ", "").lower() for entity in entities}
    except Exception:
        return set()

# load embed model (fine-tuned)
embed_tokenizer = AutoTokenizer.from_pretrained(FINETUNED_BACKBONE_PATH)
embed_model = AutoModel.from_pretrained(FINETUNED_BACKBONE_PATH).to(device).eval()

# load NLI and NER
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(device).eval()
ner_pipeline = pipeline("ner", model=NER_MODEL_NAME, tokenizer=NER_MODEL_NAME, device=0 if device.type == "cuda" else -1)

# load train dataframe (again)
df_train_full = pd.read_csv(TRAIN_CSV)
df_train_full['text'] = df_train_full['context'].fillna("") + " " + df_train_full['prompt'].fillna("") + " " + df_train_full['response'].fillna("")
y, label_names = pd.factorize(df_train_full["label"])

# A. base embeddings
F_embed = extract_embeddings(df_train_full['text'].tolist(), embed_model, embed_tokenizer, "Base Embeds")

# B. simple engineered features
F_simple = feat_engineer(df_train_full)

# C. similarity features
context_embeds = extract_embeddings(df_train_full['context'].fillna("").tolist(), embed_model, embed_tokenizer, "Context Embeds")
response_embeds = extract_embeddings(df_train_full['response'].fillna("").tolist(), embed_model, embed_tokenizer, "Response Embeds")
F_sim = np.array([cosine_similarity(context_embeds[i:i+1], response_embeds[i:i+1])[0][0] for i in range(len(df_train_full))]).reshape(-1, 1)

# D. NLI features
premise_list = df_train_full['context'].fillna("không có ngữ cảnh").tolist()
hypothesis_list = df_train_full['response'].fillna("không có phản hồi").tolist()
F_nli_parts = []
with torch.no_grad():
    for i in tqdm(range(0, len(premise_list), BATCH_SIZE_FE), desc="NLI Features"):
        batch = nli_tokenizer(premise_list[i:i+BATCH_SIZE_FE], hypothesis_list[i:i+BATCH_SIZE_FE],
                              padding=True, truncation=True, max_length=512, return_tensors="pt")
        outputs = nli_model(**{k: v.to(device) for k, v in batch.items()})
        F_nli_parts.append(torch.softmax(outputs.logits, dim=-1).cpu().numpy())
F_nli = np.vstack(F_nli_parts)

# E. NER features
new_entity_counts, entity_overlap_ratios = [], []
for _, row in tqdm(df_train_full.iterrows(), total=len(df_train_full), desc="Generating NER Features"):
    context_entities = get_entity_set(row['context'], ner_pipeline)
    response_entities = get_entity_set(row['response'], ner_pipeline)
    new_entity_counts.append(len(response_entities - context_entities))
    union = len(context_entities.union(response_entities))
    entity_overlap_ratios.append(len(context_entities.intersection(response_entities)) / union if union > 0 else 0)
F_ner = pd.DataFrame({'new_entity_count': new_entity_counts, 'entity_overlap_ratio': entity_overlap_ratios}).values

# F. Vistral probs (train)
vistral_train_probs_df = pd.read_csv(VISRAL_TRAIN_PROBS_CSV)
# merge on 'id' to align rows (assumes both have 'id' column)
df_train_with_probs = pd.merge(df_train_full.reset_index(), vistral_train_probs_df, on='id').sort_values('index').set_index('index')
prob_cols = sorted([col for col in df_train_with_probs.columns if col.startswith('prob_')])
F_vistral_probs = df_train_with_probs[prob_cols].values

# Combine features
X_final = np.hstack([F_embed, F_simple, F_sim, F_nli, F_ner, F_vistral_probs])
print("Final feature matrix shape (train):", X_final.shape)

# ---------------- Train LGBM ensemble ----------------
params = {
    'objective': 'multiclass', 'num_class': len(label_names), 'metric': 'multi_logloss',
    'verbosity': -1, 'boosting_type': 'gbdt', 'seed': 42, 'n_jobs': -1,
    'learning_rate': 0.05534380494830432, 'num_leaves': 60, 'max_depth': 5,
    'min_child_samples': 95, 'subsample': 0.6808627877925982,
    'colsample_bytree': 0.6172425473806977, 'reg_alpha': 0.004025349391603499,
    'reg_lambda': 0.0028791047868446944,
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(df_train_full), dtype=int)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_final, y)):
    print(f"Training Fold {fold+1}/5...")
    X_tr, y_tr = X_final[tr_idx], y[tr_idx]
    X_va, y_va = X_final[va_idx], y[va_idx]
    model_lgb = lgb.LGBMClassifier(**params, n_estimators=1000)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    model_path = LGBM_MODEL_PATH_TEMPLATE.format(fold)
    model_lgb.booster_.save_model(model_path)
    print(f"Fold {fold+1} model saved to {model_path}")
    oof_preds[va_idx] = model_lgb.predict(X_va)

print("Final OOF Macro-F1:", f1_score(y, oof_preds, average='macro'))

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/phobert_finetuned_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

...

Response Embeds:   0%|          | 0/438 [00:00<?, ?it/s]

NLI Features:   0%|          | 0/438 [00:00<?, ?it/s]

Generating NER Features:   0%|          | 0/7000 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final feature matrix shape (train): (7000, 783)
Training Fold 1/5...
Fold 1 model saved to /kaggle/working/lgbm_final_fold_0.txt
Training Fold 2/5...
Fold 2 model saved to /kaggle/working/lgbm_final_fold_1.txt
Training Fold 3/5...
Fold 3 model saved to /kaggle/working/lgbm_final_fold_2.txt
Training Fold 4/5...
Fold 4 model saved to /kaggle/working/lgbm_final_fold_3.txt
Training Fold 5/5...
Fold 5 model saved to /kaggle/working/lgbm_final_fold_4.txt
Final OOF Macro-F1: 0.8733188550839789


In [9]:
# ---------------- STEP 3: INFERENCE ON PRIVATE TEST ----------------
df_test = pd.read_csv(PRIVATE_TEST_CSV)
df_test['text'] = df_test['context'].fillna("") + " " + df_test['prompt'].fillna("") + " " + df_test['response'].fillna("")

# A. Base embeddings
F_test_embed = extract_embeddings(df_test['text'].tolist(), embed_model, embed_tokenizer, "Test Base Embeds")

# B. simple features
F_test_simple = feat_engineer(df_test)

# C. similarity features
test_context_embeds = extract_embeddings(df_test['context'].fillna("").tolist(), embed_model, embed_tokenizer, "Test Context Embeds")
test_response_embeds = extract_embeddings(df_test['response'].fillna("").tolist(), embed_model, embed_tokenizer, "Test Response Embeds")
F_test_sim = np.array([cosine_similarity(test_context_embeds[i:i+1], test_response_embeds[i:i+1])[0][0] for i in range(len(df_test))]).reshape(-1, 1)

# D. NLI features
test_premise = df_test['context'].fillna("không có ngữ cảnh").tolist()
test_hypothesis = df_test['response'].fillna("không có phản hồi").tolist()
F_test_nli_parts = []
with torch.no_grad():
    for i in tqdm(range(0, len(test_premise), BATCH_SIZE_FE), desc="Test NLI Features"):
        batch = nli_tokenizer(test_premise[i:i+BATCH_SIZE_FE], test_hypothesis[i:i+BATCH_SIZE_FE],
                              padding=True, truncation=True, max_length=512, return_tensors="pt")
        outputs = nli_model(**{k: v.to(device) for k, v in batch.items()})
        F_test_nli_parts.append(torch.softmax(outputs.logits, dim=-1).cpu().numpy())
F_test_nli = np.vstack(F_test_nli_parts)

# E. NER features for test
test_new_entity_counts, test_entity_overlap_ratios = [], []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating Test NER Features"):
    context_entities = get_entity_set(row['context'], ner_pipeline)
    response_entities = get_entity_set(row['response'], ner_pipeline)
    test_new_entity_counts.append(len(response_entities - context_entities))
    union = len(context_entities.union(response_entities))
    test_entity_overlap_ratios.append(len(context_entities.intersection(response_entities)) / union if union > 0 else 0)
F_test_ner = pd.DataFrame({'new_entity_count': test_new_entity_counts, 'entity_overlap_ratio': test_entity_overlap_ratios}).values

# F. Vistral probs for test
vistral_test_probs_df = pd.read_csv(VISRAL_TEST_PROBS_CSV)
df_test_with_probs = pd.merge(df_test.reset_index(), vistral_test_probs_df, on='id').sort_values('index').set_index('index')
prob_cols_test = sorted([col for col in df_test_with_probs.columns if col.startswith('prob_')])
F_test_vistral_probs = df_test_with_probs[prob_cols_test].values

# Combine test features
X_test_final = np.hstack([F_test_embed, F_test_simple, F_test_sim, F_test_nli, F_test_ner, F_test_vistral_probs])
print("Final test feature matrix shape:", X_test_final.shape)

# Predict with saved ensemble
all_test_probas = []
for fold in range(5):
    model_path = LGBM_MODEL_PATH_TEMPLATE.format(fold)
    bst = lgb.Booster(model_file=model_path)
    probas = bst.predict(X_test_final)
    all_test_probas.append(probas)

final_preds = np.argmax(np.mean(all_test_probas, axis=0), axis=1)
pred_labels = [label_names[i] for i in final_preds]

submission = pd.DataFrame({"id": df_test["id"], "predict_label": pred_labels})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
print(submission.head())

Test Base Embeds:   0%|          | 0/125 [00:00<?, ?it/s]

Test Context Embeds:   0%|          | 0/125 [00:00<?, ?it/s]

Test Response Embeds:   0%|          | 0/125 [00:00<?, ?it/s]

Test NLI Features:   0%|          | 0/125 [00:00<?, ?it/s]

Generating Test NER Features:   0%|          | 0/2000 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


Final test feature matrix shape: (2000, 783)
Submission saved to submission.csv
                                     id predict_label
0  ef35e7a1-766f-4455-b349-09084a1f56ba            no
1  85aac4aa-e53c-4d01-bdcf-e8b27a2cd9bc     intrinsic
2  cd056e1b-51f6-4adf-939c-e742645437bf            no
3  6cc80aa4-44db-4366-8cb6-16453aa8ecbf     intrinsic
4  948849cb-0c83-4dc1-b5ae-aaa585a8d528     intrinsic
